In [5]:
#用python加载数据到MySQL
import pandas as pd
import mysql.connector
conn = mysql.connector.connect(host='localhost', user='root', password='123456', database='loan_db')
cursor = conn.cursor()
data = pd.read_csv('application_train.csv')
data = data[['SK_ID_CURR', 'TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']].fillna(0)
for index, row in data.iterrows():
    cursor.execute("INSERT INTO applications (SK_ID_CURR, TARGET, AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY) VALUES (%s, %s, %s, %s, %s)",
    (row['SK_ID_CURR'], row['TARGET'], row['AMT_INCOME_TOTAL'], row['AMT_CREDIT'], row['AMT_ANNUITY']))
conn.commit()
conn.close()

In [7]:
#用SQL查询分析
#查询平均收入和高贷款违约
conn = mysql.connector.connect(host='localhost', user='root', password='123456', database='loan_db')
cursor = conn.cursor()
cursor.execute("SELECT AVG(AMT_INCOME_TOTAL) FROM applications WHERE TARGET = 1")
avg_income_default = cursor.fetchone()
print(avg_income_default)
cursor.execute("SELECT COUNT(*) FROM applications WHERE TARGET = 1 AND AMT_CREDIT > 500000")
high_credit_default = cursor.fetchone()
print(high_credit_default)
conn.close()

(165611.76090634443,)
(12220,)


In [2]:
import mysql.connector

conn = mysql.connector.connect(host='localhost', user='root', password='123456', database='loan_db')
cursor = conn.cursor()

# 1. 整体违约率
cursor.execute("SELECT ROUND(AVG(TARGET) * 100, 2) FROM applications")
overall_default = cursor.fetchone()[0]
print(f"整体违约率: {overall_default}%")

# 2. 高额贷款违约率 (AMT_CREDIT > 500000)
cursor.execute("SELECT ROUND(AVG(TARGET) * 100, 2) FROM applications WHERE AMT_CREDIT > 500000")
high_credit_default = cursor.fetchone()[0]
print(f"高额贷款违约率: {high_credit_default}%")

# 3. 高风险群体违约率 (高额贷款 + 低收入)
cursor.execute("""
  SELECT ROUND(AVG(TARGET) * 100, 2) 
  FROM applications 
  WHERE AMT_CREDIT > 500000 AND AMT_INCOME_TOTAL < 100000
""")
high_risk_default = cursor.fetchone()[0]
print(f"高风险群体违约率: {high_risk_default}%")

conn.close()

整体违约率: 8.07%
高额贷款违约率: 7.73%
高风险群体违约率: 8.35%


In [4]:
#用Scikit-learn逻辑回归模型预测违约风险，并计算测试集准确率
import pandas as pd
import mysql.connector
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#连接数据库，把数据读成pandas表格
conn = mysql.connector.connect(host='localhost', user='root', password='123456', database='loan_db')
df = pd.read_sql("SELECT AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, TARGET FROM applications", conn)
conn.close()

#准备特征和标签
X = df[['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']]
y = df['TARGET']

#把数据分成训练集和测试集（80%训练，20%测试）
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#创建逻辑回归模型
model = LogisticRegression(max_iter=1000)

#用训练数据训练模型
model.fit(X_train, y_train)

#用测试数据做预测
predictions = model.predict(X_test)

# 计算并打印测试集准确率
accuracy = round(accuracy_score(y_test, predictions) * 100, 2)
print(f"模型在测试集的准确率: {accuracy}%")

C:\Users\huili\AppData\Local\Temp\ipykernel_10444\3964618084.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, TARGET FROM applications", conn)


模型在测试集的准确率: 91.95%
